# Getting Started with pyth-pandas

Every endpoint returns a **pandas DataFrame**. This notebook walks through the basics:
fetching latest prices, converting raw mantissas to human-readable values, and pulling
historical feed snapshots.

**You need a Pyth Pro access token** — request one via the [onboarding form](https://docs.pyth.network/price-feeds/pro)
and set `PYTH_API_KEY` in your environment (or in a `.env` file at the notebook root).


In [ ]:
!pip install -q pyth-pandas matplotlib python-dotenv

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from pyth_pandas import PythPandas

client = PythPandas()

## Latest prices

`fetch_latest_prices` returns one row per feed. Request any subset of the documented
[feed properties](https://docs.pyth.network/price-feeds/pro/api) — you only pay for what you ask for.

In [ ]:
df = client.fetch_latest_prices(
    symbols=["Crypto.BTC/USD", "Crypto.ETH/USD", "Crypto.SOL/USD"],
    properties=[
        "price",
        "confidence",
        "exponent",
        "publisherCount",
        "bestBidPrice",
        "bestAskPrice",
    ],
    formats=[],
)
df

## Human-readable prices

`price` and `confidence` are raw mantissas — multiply by `10 ** exponent` to get
the actual decimal value. Best-bid/best-ask use the same exponent.

In [ ]:
scale = 10 ** df["exponent"]

df["humanPrice"] = df["price"] * scale
df["humanBid"] = df["bestBidPrice"] * scale
df["humanAsk"] = df["bestAskPrice"] * scale
df["humanConfidence"] = df["confidence"] * scale

df[["priceFeedId", "humanPrice", "humanBid", "humanAsk", "humanConfidence", "publisherCount"]]

## Update timestamp

The parsed payload's wall-clock time (microsecond resolution) is attached as `df.attrs`.
This is the `timestampUs` from the Pyth Pro response.

In [ ]:
import pandas as pd

ts = pd.Timestamp(int(df.attrs["timestampUs"]), unit="us", tz="UTC")
print("Update published at:", ts)

## Channels

The `channel` argument controls update cadence:
- `real_time` — emitted as soon as publishers push (default)
- `fixed_rate@50ms` / `@200ms` / `@1000ms` — aggregated to the specified interval

The fixed-rate channels are useful when you need deterministic replayable updates.

In [ ]:
fixed = client.fetch_latest_prices(
    symbols=["Crypto.BTC/USD"],
    properties=["price", "exponent", "feedUpdateTimestamp"],
    channel="fixed_rate@1000ms",
    formats=[],
)
fixed

## On-chain payloads

Add `formats=["evm", "solana", "leEcdsa", "leUnsigned"]` to get signed blobs suitable for
on-chain verification. They arrive attached to `df.attrs`, keyed by format name.

In [ ]:
with_evm = client.fetch_latest_prices(
    symbols=["Crypto.BTC/USD"],
    properties=["price", "exponent"],
    formats=["evm", "solana"],
)
print("EVM payload (hex, truncated):", with_evm.attrs["evm"]["data"][:100], "...")
print("Solana payload:", with_evm.attrs["solana"]["data"][:100], "...")

## Raw JsonUpdate

If you want the full response dict (no DataFrame wrapping), use the `_raw` variants.

In [ ]:
payload = client.fetch_latest_prices_raw(
    symbols=["Crypto.BTC/USD"],
    properties=["price", "exponent"],
    formats=["evm"],
)
list(payload.keys())

## Next up

- **`02_historical_prices.ipynb`** — query feed values at specific timestamps and plot OHLC.
- **`03_websocket_streaming.ipynb`** — subscribe to the real-time stream and consume updates as they arrive.